<a href="https://colab.research.google.com/github/Maximi652/efficient-slm-architectures/blob/main/LotteryTicketSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==== SETUP ====
from google.colab import drive
drive.mount('/content/drive')

!pip install --upgrade transformers accelerate tqdm

import torch
import copy
import json
import re
import gc
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn.utils import prune
import torch.nn.functional as F

# === PARAMETER & PFADEN ===
MODEL_PATH    = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B"
INPUT_JSON    = "/content/drive/MyDrive/Colab Notebooks/12B_combined_golden.json"
PRUNED_MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Qwen3-4B-LTH"
OUTPUT_JSON   = "/content/drive/MyDrive/Colab Notebooks/lth-qwen3-4b_testresults.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_ITER = 5          # Anzahl der LTH-Pruning-Iterationen
PRUNE_RATE = 0.1      # Prune-Rate pro Iteration
EPOCHS_PER_ITER = 3   # Anzahl Epochen pro Iteration (für Demo niedrig, bei echten Experimenten 2–3+)
BATCH_SIZE = 2        # Training batch size
TRAINSET_SIZE = 16
MAX_PROMPT = 256
MAX_TARGET = 64

# ==== 1. Modell & Tokenizer laden, Initialisierung speichern ====
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float32,
    device_map='cpu'
)
model.train()

In [ ]:
# ==== 2. Nur Linear-Layer-Initialisierung auf CPU sichern ====
init_state = {}
for name, param in model.named_parameters():
    if "weight" in name and param.dim() > 1:
        init_state[name] = param.cpu().clone()

# ==== 3. Trainingsdaten klein halten ====
def get_training_examples(json_file, N=16):
    with open(json_file, "r", encoding="utf-8") as f:
        questions = json.load(f)['questions']
    texts = []
    for q in questions[:N]:
        qtxt = q['body']
        ctx = "\n".join([s['text'] for s in q.get('snippets', [])[:1]])  # nur 1 Context
        target = q.get('exact_answer', "")
        if isinstance(target, list):
            target = ", ".join([str(x) for x in target])
        if ctx:
            prompt = f"Question: {qtxt}\nContext:\n{ctx}\nAnswer:"
        else:
            prompt = f"Question: {qtxt}\nAnswer:"
        texts.append((prompt, str(target)))
    return texts

train_examples = get_training_examples(INPUT_JSON, N=TRAINSET_SIZE)

# ==== 4. Supervised Data Helper (Prompt+Target) ====
def make_supervised_data(examples, tokenizer, max_length=96, max_target_length=16):
    input_ids_list, labels_list = [], []
    for prompt, target in examples:
        full_text = prompt + " " + target
        tokenized = tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length + max_target_length,
            padding="max_length"
        )
        input_ids = tokenized['input_ids'][0]
        prompt_ids = tokenizer(prompt, truncation=True, max_length=max_length, add_special_tokens=False)['input_ids']
        label = input_ids.clone()
        label[:len(prompt_ids)] = -100
        input_ids_list.append(input_ids)
        labels_list.append(label)
    input_ids = torch.stack(input_ids_list)
    labels = torch.stack(labels_list)
    return input_ids, labels

# ==== 5. Speicherfreundliches Training ====
def finetune(model, tokenizer, train_data, num_epochs=1, batch_size=1, lr=1e-5, max_length=96, max_target_length=16):
    model.train()
    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = max(1, (len(train_data) // batch_size) * num_epochs)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    losses = []
    for epoch in range(num_epochs):
        pbar = tqdm(range(0, len(train_data), batch_size), desc=f'Epoch {epoch+1}/{num_epochs}')
        for i in pbar:
            batch = train_data[i:i+batch_size]
            input_ids, labels = make_supervised_data(batch, tokenizer, max_length, max_target_length)
            input_ids = input_ids.to('cpu')
            labels = labels.to('cpu')
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix(loss=loss.item())
            losses.append(loss.item())
            # RAM schonen
            del input_ids, labels, outputs, loss
            torch.cuda.empty_cache()
            gc.collect()
    del optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    return losses

# ==== 6. RAM-schonendes LTH-Pruning ====
def lth_prune(model, amount):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            prune.l1_unstructured(module, name="weight", amount=amount)
            prune.remove(module, "weight")
    return model

def lth_reset(model, init_state):
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in init_state and param.dim() > 1:
                # alles auf CPU holen
                p_data = param.detach().cpu()
                init_data = init_state[name]
                mask = (p_data != 0)
                # nur die maskierten Gewichte ersetzen
                p_data[mask] = init_data[mask]
                # zurück aufs Device
                param.data.copy_(p_data.to(param.device))
    torch.cuda.empty_cache()
    gc.collect()

# ==== 7. OpenLTH-Style LTH-Loop ====
for it in range(NUM_ITER):
    print(f"\n=== LTH Iteration {it+1}/{NUM_ITER} ===")
    # a) Training
    finetune(model, tokenizer, train_examples, num_epochs=EPOCHS_PER_ITER, batch_size=BATCH_SIZE, max_length=MAX_PROMPT, max_target_length=MAX_TARGET)
    torch.cuda.empty_cache(); gc.collect()
    # b) Pruning
    lth_prune(model, PRUNE_RATE)
    print(f"Pruned {PRUNE_RATE*100:.1f}% of Linear weights.")
    # c) Reset: Nur Lineare Layer (per Maske), aus CPU-Init!
    lth_reset(model, init_state)
    print("Linear-Layer-Weights per Mask auf Initialisierung gesetzt.")
    # <<-- HIER: Modell auf Platte speichern und aus RAM löschen
    tmp_path = f"/tmp/lth_iter{it+1}.bin"
    model.save_pretrained(tmp_path)
    del model
    torch.cuda.empty_cache(); gc.collect()
    # Und dann direkt reloaden:
    model = AutoModelForCausalLM.from_pretrained(tmp_path, torch_dtype="auto", device_map=DEVICE)
    model.train()

# ==== 8. Finales Feintuning ====
print("\n==== Feintuning des finalen Winning Tickets ====")
finetune(model, tokenizer, train_examples, num_epochs=EPOCHS_PER_ITER, batch_size=BATCH_SIZE, max_length=MAX_PROMPT, max_target_length=MAX_TARGET)
torch.cuda.empty_cache(); gc.collect()

# ==== 9. Modell speichern ====
model.save_pretrained(PRUNED_MODEL_PATH)
print(f"Gepruntes LTH-Modell gespeichert unter {PRUNED_MODEL_PATH}")

# ==== 10. Metriken zeigen ====
def compute_pruned_stats(model):
    total_params = 0
    nonzero_params = 0
    for name, param in model.named_parameters():
        if param.requires_grad and param.dim() > 1 and "weight" in name:
            total_params += param.numel()
            nonzero_params += torch.count_nonzero(param).item()
    zero_params = total_params - nonzero_params
    sparsity = 100.0 * zero_params / total_params
    compression_ratio = total_params / nonzero_params if nonzero_params > 0 else float("inf")
    print(f"Gesamtparameter: {total_params:,}")
    print(f"Aktive Parameter (<> 0): {nonzero_params:,}")
    print(f"Sparsity: {sparsity:.2f}%")
    print(f"Komprimierungsrate: {compression_ratio:.2f}x")
    return total_params, nonzero_params, sparsity, compression_ratio

compute_pruned_stats(model)